# Lab 7 FDSAI (2025-2026) – Normality: graphical methods and tests
Run all cells from top to bottom. The four datasets of the lab sheet are typed directly in the notebook (no file upload needed). Significance level: **α = 0.05** unless stated otherwise.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

## Data

In [ ]:
# Dataset [1] - Height of 50 athletes (cm)
height = np.array([187.3, 177.8, 181.9, 190.7, 188.1, 168.2, 181.7, 173.9, 174.3, 177.9,
                   176.0, 185.2, 180.3, 175.9, 178.1, 177.3, 185.5, 173.6, 177.2, 169.0,
                   157.1, 179.6, 181.1, 169.8, 190.9, 164.8, 175.3, 173.7, 185.7, 185.3,
                   176.1, 177.6, 168.8, 161.1, 172.6, 176.1, 183.6, 183.4, 172.3, 172.9,
                   167.7, 165.1, 163.1, 188.7, 171.4, 171.9, 166.2, 180.4, 163.7, 173.5])

# Dataset [2] - Reaction times (ms) of 40 participants
reaction = np.array([367, 210, 214, 187, 304, 138, 378, 202, 265, 230,
                     261, 206, 199, 278, 242, 223, 232, 279, 244, 193,
                     219, 247, 245, 212, 206, 237, 249, 206, 257, 281,
                     284, 268, 215, 218, 204, 246, 205, 230, 222, 209], dtype=float)

# Dataset [3] - Exam scores of two student groups
scores_A = np.array([71.7, 74.5, 57.9, 88.1, 60.7, 68.3, 79.0, 65.0, 66.5, 67.7,
                     79.4, 93.3, 75.3, 66.1, 79.3, 70.2, 74.8, 84.4, 69.0, 75.1,
                     68.0, 73.7, 77.1, 67.1, 72.3, 73.1, 69.9, 65.5, 63.6, 73.8])
scores_B = np.array([67.3, 92.3, 45.7, 71.1, 73.7, 83.6, 75.0, 61.6, 70.0, 75.4,
                     59.4, 72.5, 80.3, 75.4, 81.8, 57.4, 49.2, 50.5, 81.7, 78.7,
                     69.9, 68.3, 71.6, 65.1, 75.7, 48.1, 57.6, 79.7])

# Dataset [4] - Waiting time (min)
wait = np.array([8.0, 12.3, 3.4, 7.2, 22.3, 22.7, 1.3, 2.3, 0.5, 5.8,
                 4.4, 13.6, 3.9, 8.4, 2.4, 4.0, 5.9, 0.5, 1.0, 12.4,
                 20.4, 11.1, 13.1, 6.7, 11.3, 11.3, 4.8, 1.4, 1.4, 7.7,
                 11.2, 0.5, 7.7, 4.4, 4.3, 37.3, 0.2, 6.4, 7.9, 7.7,
                 6.6, 1.6, 4.3, 5.2, 2.5, 10.5, 11.1, 4.4, 6.6, 0.3])

# Sanity check: sample sizes must match the lab sheet (50, 40, 30 + 28, 50)
print(len(height), len(reaction), len(scores_A), len(scores_B), len(wait))

## Helper functions
`hist_and_qq` draws the histogram (with the normal curve fitted with the sample mean and standard deviation) and the normal Q-Q plot; `describe_shape` computes the skewness and kurtosis; `interpret_p` writes the decision of a test.

In [ ]:
ALPHA = 0.05

def hist_and_qq(x, variable, title):
    # Histogram with the fitted normal curve (left) and normal Q-Q plot (right)
    x = np.asarray(x, dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].hist(x, bins="auto", density=True, color="lightsteelblue", edgecolor="white", label="Sample")
    grid = np.linspace(x.min() - x.std(), x.max() + x.std(), 300)
    axes[0].plot(grid, stats.norm.pdf(grid, x.mean(), x.std(ddof=1)), color="tab:red", linewidth=2,
                 label=f"Normal(mean={x.mean():.1f}, sd={x.std(ddof=1):.1f})")
    axes[0].axvline(x.mean(), color="gray", linestyle="--", label="Mean")
    axes[0].axvline(np.median(x), color="black", linestyle=":", label="Median")
    axes[0].set_title(f"Histogram - {title}"); axes[0].set_xlabel(variable); axes[0].set_ylabel("Density")
    axes[0].legend(fontsize=9)

    # Q-Q plot: sample quantiles against theoretical normal quantiles; the red line is the reference
    stats.probplot(x, dist="norm", plot=axes[1])
    axes[1].set_title(f"Normal Q-Q plot - {title}")
    axes[1].set_xlabel("Theoretical quantiles"); axes[1].set_ylabel(f"Sample quantiles ({variable})")
    axes[1].grid(True, linestyle="--", alpha=0.35)
    plt.tight_layout(); plt.show()

def describe_shape(x, name="sample"):
    # Skewness and (excess) kurtosis of a sample
    x = np.asarray(x, dtype=float)
    return {"Sample": name, "n": len(x), "Mean": x.mean(), "Median": np.median(x), "SD": x.std(ddof=1),
            "Skewness": stats.skew(x, bias=False), "Excess kurtosis": stats.kurtosis(x, bias=False)}

def interpret_p(p, alpha, name="the data"):
    # Decision of a normality test (H0: normal distribution) at significance level alpha
    if p < alpha:
        return f"alpha = {alpha}: p = {p:.4g} < {alpha} -> reject H0 ({name} are NOT normally distributed)"
    return f"alpha = {alpha}: p = {p:.4g} >= {alpha} -> fail to reject H0 (no evidence against normality for {name})"

## Exercise 1 – Visual analysis of the heights of 50 athletes

### 1a) Histogram and 1b) Q-Q plot

In [ ]:
d = describe_shape(height, "Height (cm)")
display(pd.DataFrame([d]).round(3))
hist_and_qq(height, "height (cm)", "athletes' height")

**How to read the Q-Q plot.** Each point compares a sample quantile (vertical axis) with the quantile expected under a normal distribution (horizontal axis). If the data are normal, the points lie on the straight red line. Curvature = skewness, an S-shape or points leaving the line at the ends = tail deviations.

### 1c) Answers based on the visual analysis

In [ ]:
# Numerical support for the visual analysis (the lab asks for a visual answer, these numbers only back it up)
x = height
print(f"Skewness = {stats.skew(x, bias=False):.3f}   (0 for a symmetric distribution)")
print(f"Excess kurtosis = {stats.kurtosis(x, bias=False):.3f}   (0 for a normal distribution)")
z = (np.sort(x) - x.mean()) / x.std(ddof=1)
print(f"Smallest value: {x.min()} cm (z = {z[0]:.2f}),  largest value: {x.max()} cm (z = {z[-1]:.2f})")
print(f"Values with |z| > 2: {(np.abs(z) > 2).sum()} of {len(x)}  (about 5% = {0.05 * len(x):.1f} expected under normality)")
w, p = stats.shapiro(x)
print(f"Shapiro-Wilk (only as a check): W = {w:.4f}, p = {p:.4f}")

**i) Does the data appear approximately normal?** **Yes.** The histogram is unimodal, roughly symmetric and bell-shaped, with the mean (≈ 176.0 cm) and the median (≈ 176.1 cm) practically equal, and in the Q-Q plot the points follow the reference line closely over the whole range. The Shapiro–Wilk test used only as a check (p ≈ 0.88) does not contradict this.

**ii) Are there deviations at the tails?** Only **minor** ones. The two most extreme observations (157.1 cm and 190.9 cm) are within about ±2.4 standard deviations of the mean, and the largest value lies a little below the line (the upper tail is slightly shorter than a normal one). The excess kurtosis is slightly negative (≈ −0.41), i.e. the tails are, if anything, slightly lighter than normal. With n = 50 these small deviations are within normal sampling variation.

**iii) If not normal, what type of skewness is visible?** The data are close to normal, so there is **no meaningful skewness**: the sample skewness is ≈ −0.14, a very slight tendency to a longer left tail, which is negligible.

## Exercise 2 – Shapiro–Wilk test for the reaction times (n = 40)

### 2a) Shapiro–Wilk test

In [ ]:
w, p = stats.shapiro(reaction)
print(f"n = {len(reaction)}, mean = {reaction.mean():.1f} ms, median = {np.median(reaction):.1f} ms, sd = {reaction.std(ddof=1):.1f} ms")
print(f"Shapiro-Wilk: W = {w:.4f}, p-value = {p:.6f}")

### 2b) Hypotheses
- **H0:** the reaction times come from a normally distributed population.
- **H1:** the reaction times do **not** come from a normally distributed population.

(The test statistic W is close to 1 for normal data; a small p-value is evidence **against** normality.)

### 2c) p-value and interpretation for α = 0.05 and α = 0.01

In [ ]:
print(interpret_p(p, 0.05, "the reaction times"))
print(interpret_p(p, 0.01, "the reaction times"))

**Interpretation.** The p-value (≈ 0.0018) is smaller than both 0.05 and 0.01, so H0 is rejected at **both** significance levels: there is strong evidence that the reaction times are not normally distributed. The decision does not depend on which of the two usual levels is chosen.

### 2d) Can the reaction times be modeled with a normal distribution?

In [ ]:
d = describe_shape(reaction, "Reaction time (ms)")
display(pd.DataFrame([d]).round(3))
hist_and_qq(reaction, "reaction time (ms)", "reaction times")

# Outliers by the IQR rule (they explain most of the deviation from normality)
q1, q3 = np.percentile(reaction, [25, 75]); iqr = q3 - q1
outliers = reaction[(reaction < q1 - 1.5 * iqr) | (reaction > q3 + 1.5 * iqr)]
print("Outliers (IQR rule):", np.sort(outliers))

# Does a log transformation help? (reaction times are often approximately log-normal)
w_log, p_log = stats.shapiro(np.log(reaction))
print(f"Shapiro-Wilk on ln(reaction time): W = {w_log:.4f}, p = {p_log:.4f}")

**Answer: no.** The Shapiro–Wilk test rejects normality at α = 0.05 and at α = 0.01. The distribution is clearly **right-skewed** (skewness ≈ 1.2, excess kurtosis ≈ 2.9, mean 237.8 ms > median 230 ms) with a few very slow reactions (367 ms and 378 ms are outliers by the IQR rule), which is typical for reaction times. A simple log transformation improves the shape but the result is still borderline (p ≈ 0.037: rejected at 0.05, not at 0.01). A normal model is therefore not appropriate; skewed models (log-normal, gamma, ex-Gaussian) or nonparametric methods are better choices.

## Exercise 3 – Shapiro–Wilk versus Kolmogorov–Smirnov (exam scores of groups A and B)

### 3a) Both tests for each group
For both tests H0: the scores of the group are normally distributed; H1: they are not. The Kolmogorov–Smirnov test is applied with the **normal reference parameters estimated from the sample** (mean and standard deviation of the group). Because the parameters are estimated from the same data, the standard KS p-values are too large (the test becomes too conservative), so the Lilliefors-corrected p-value is also shown as a reference.

In [ ]:
def ks_normal(x):
    # Kolmogorov-Smirnov test against N(mean, sd) with the parameters estimated from the sample
    return stats.kstest(x, stats.norm(loc=np.mean(x), scale=np.std(x, ddof=1)).cdf)

try:
    from statsmodels.stats.diagnostic import lilliefors     # KS with estimated parameters (corrected p-value)
except ImportError:
    lilliefors = None

rows = []
for name, x in [("A", scores_A), ("B", scores_B)]:
    sw = stats.shapiro(x)
    ks = ks_normal(x)
    row = {"Group": name, "n": len(x), "Mean": x.mean(), "SD": x.std(ddof=1),
           "Shapiro W": sw.statistic, "Shapiro p": sw.pvalue,
           "KS D": ks.statistic, "KS p": ks.pvalue}
    if lilliefors is not None:
        row["Lilliefors p"] = lilliefors(x, dist="norm", pvalmethod="approx")[1]
    row["SW decision (0.05)"] = "reject" if sw.pvalue < ALPHA else "do not reject"
    row["KS decision (0.05)"] = "reject" if ks.pvalue < ALPHA else "do not reject"
    rows.append(row)
display(pd.DataFrame(rows).round(4))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for j, (name, x) in enumerate([("A", scores_A), ("B", scores_B)]):
    axes[0, j].hist(x, bins=10, density=True, color="lightsteelblue", edgecolor="white")
    grid = np.linspace(x.min() - 10, x.max() + 10, 300)
    axes[0, j].plot(grid, stats.norm.pdf(grid, x.mean(), x.std(ddof=1)), color="tab:red", linewidth=2)
    axes[0, j].set_title(f"Group {name}: histogram"); axes[0, j].set_xlabel("score")
    stats.probplot(x, dist="norm", plot=axes[1, j]); axes[1, j].set_title(f"Group {name}: Q-Q plot")
plt.tight_layout(); plt.show()

### 3b) Comparison of the two tests

**i) Do both tests lead to the same conclusion?** Yes. For both groups the p-values of both tests are far above 0.05 (Shapiro–Wilk: p ≈ 0.38 for A and p ≈ 0.29 for B; Kolmogorov–Smirnov: p ≈ 0.76 for A and p ≈ 0.69 for B), so **H0 is not rejected in either group**: the scores of both groups are compatible with a normal distribution. Note that "not rejected" does not prove normality, especially for n ≈ 30.

**ii) and iii)** The two questions about power and tail sensitivity are answered with a small simulation (2000 samples for each scenario): the table shows the percentage of samples for which each test rejects H0. For normal data it measures the type I error (should be about 5%); for non-normal data it measures the **power**.

In [ ]:
# Simulation: how often does each test reject H0 (alpha = 0.05) when the data are NOT normal?
# For a normal sample the rejection rate is the type I error (should be near 5%).
rng = np.random.default_rng(1)
N_SIM = 2000

def rejection_rates(sampler, n):
    rej = {"Shapiro-Wilk": 0, "Kolmogorov-Smirnov": 0, "Lilliefors": 0}
    for _ in range(N_SIM):
        x = sampler(n)
        rej["Shapiro-Wilk"] += stats.shapiro(x).pvalue < ALPHA
        rej["Kolmogorov-Smirnov"] += ks_normal(x).pvalue < ALPHA
        if lilliefors is not None:
            rej["Lilliefors"] += lilliefors(x, dist="norm", pvalmethod="approx")[1] < ALPHA
    return {k: v / N_SIM for k, v in rej.items()}

scenarios = [
    ("Normal (type I error), n = 20", lambda n: rng.normal(0, 1, n), 20),
    ("Skewed: exponential, n = 20", lambda n: rng.exponential(1, n), 20),
    ("Skewed: log-normal, n = 20", lambda n: rng.lognormal(0, 0.6, n), 20),
    ("Light tails: uniform, n = 20", lambda n: rng.uniform(0, 1, n), 20),
    ("Heavy tails: Student t (3 df), n = 50", lambda n: rng.standard_t(3, n), 50),
]
power = pd.DataFrame({label: rejection_rates(s, n) for label, s, n in scenarios}).T
display((power * 100).round(1).rename_axis("Scenario (rejection rate, %)"))

**ii) Which test is more powerful for small samples?** The **Shapiro–Wilk** test. In the simulation with n = 20 it rejects non-normal data much more often (e.g. 83% for the exponential and 66% for the log-normal distribution) than the Kolmogorov–Smirnov test with estimated parameters (≈ 3–4%), and also more often than the Lilliefors-corrected KS test (57% and 45%). The plain KS test with estimated parameters is extremely conservative (it almost never rejects, even under non-normal data); it is designed for a completely specified reference distribution, so with estimated parameters the Lilliefors version should be used.

**iii) Which one is more sensitive to deviations in the tails?** The **Shapiro–Wilk** test. The KS statistic is the maximum distance between the empirical and the theoretical distribution functions, and this distance is largest near the center of the distribution, where the two curves can separate most; in the tails both curves are close to 0 or to 1, so tail deviations hardly change the statistic. The Shapiro–Wilk statistic is based on the ordered values (it is related to the correlation in the Q-Q plot), so it reacts to extreme values. In the simulation, for heavy-tailed data (Student t with 3 degrees of freedom, n = 50) Shapiro–Wilk rejects in ≈ 63% of the samples, the Lilliefors test in ≈ 47% and the plain KS test only in ≈ 10%.

## Exercise 4 – Skewness, kurtosis, D'Agostino's K² test and transformations (waiting times)

### 4a) Skewness and kurtosis of the raw data

In [ ]:
print(f"n = {len(wait)}, min = {wait.min()}, max = {wait.max()}")
raw = describe_shape(wait, "Raw waiting time")
display(pd.DataFrame([raw]).round(3))
print(f"Skewness (Fisher-Pearson, unbiased) = {stats.skew(wait, bias=False):.4f}")
print(f"Excess kurtosis (unbiased)           = {stats.kurtosis(wait, bias=False):.4f}  "
      f"(Pearson kurtosis = {stats.kurtosis(wait, fisher=False, bias=False):.4f})")
hist_and_qq(wait, "waiting time (min)", "raw waiting time")

**Interpretation.** The skewness (≈ 2.13) is strongly positive: the distribution has a long right tail (mean 7.4 min versus median 6.15 min, with a maximum of 37.3 min). The excess kurtosis (≈ 6.5; Pearson kurtosis ≈ 9.5) is far above 0 (the normal value), so the distribution is very heavy-tailed. Both indicators point to a strong departure from normality.

### 4b) D'Agostino's K² test on the raw data
H0: the waiting times are normally distributed; H1: they are not. The statistic is $K^2=z_s^2+z_k^2$, where $z_s$ and $z_k$ are the standardized skewness and kurtosis; under H0 it follows approximately a chi-square distribution with 2 degrees of freedom.

In [ ]:
# D'Agostino's K^2 test combines the skewness test (z_s) and the kurtosis test (z_k): K2 = z_s^2 + z_k^2
k2, p_k2 = stats.normaltest(wait)
z_s = stats.skewtest(wait); z_k = stats.kurtosistest(wait)
print(f"Skewness test:  z = {z_s.statistic:.3f}, p = {z_s.pvalue:.6f}")
print(f"Kurtosis test:  z = {z_k.statistic:.3f}, p = {z_k.pvalue:.6f}")
print(f"D'Agostino K^2 = {k2:.3f}, p = {p_k2:.3g}")
print(interpret_p(p_k2, 0.05, "the waiting times"))
print(interpret_p(p_k2, 0.01, "the waiting times"))

**Interpretation.** K² ≈ 37.3 and p ≈ 8·10⁻⁹, so H0 is rejected at α = 0.05 and α = 0.01. Both components are significant: the skewness test (z ≈ 4.7) and the kurtosis test (z ≈ 3.8). The raw waiting times are clearly not normal.

### 4c) Transformations
The waiting times are all positive (minimum 0.2 min), so the logarithm can be applied. Besides the square root and the natural logarithm (suggested by the lab sheet), the cube root, log(1 + x) and two parametric transformations (Box–Cox and Yeo–Johnson, where the exponent λ is chosen by maximum likelihood to make the data as normal as possible) are tried.

In [ ]:
# Candidate transformations (all waiting times are > 0, so the log is defined)
bc_values, bc_lambda = stats.boxcox(wait)             # Box-Cox: lambda estimated by maximum likelihood
yj_values, yj_lambda = stats.yeojohnson(wait)         # Yeo-Johnson: same idea, works also for x <= 0

transformed = {
    "Raw (no transformation)": wait,
    "Square root": np.sqrt(wait),
    "Cube root": np.cbrt(wait),
    "Natural log": np.log(wait),
    "log(1 + x)": np.log1p(wait),
    f"Box-Cox (lambda = {bc_lambda:.2f})": bc_values,
    f"Yeo-Johnson (lambda = {yj_lambda:.2f})": yj_values,
}
SIMPLE = ["Square root", "Cube root", "Natural log", "log(1 + x)"]    # transformations without an estimated parameter

### 4d) Normality tests after the transformations
D'Agostino's K² test is repeated for every transformation, together with the skewness, the kurtosis and the Shapiro–Wilk test as a second opinion.

In [ ]:
rows = []
for name, v in transformed.items():
    k2_v, p_k2_v = stats.normaltest(v)
    w_v, p_sw_v = stats.shapiro(v)
    rows.append({"Transformation": name,
                 "Skewness": stats.skew(v, bias=False), "Excess kurtosis": stats.kurtosis(v, bias=False),
                 "K2": k2_v, "K2 p-value": p_k2_v, "Shapiro p-value": p_sw_v,
                 "Normal at 0.05 (K2)": "yes" if p_k2_v >= ALPHA else "no"})
results = pd.DataFrame(rows).set_index("Transformation")
display(results.round(4))

### 4e) Comparison before / after

In [ ]:
# Which transformation improved normality the most? (highest p-value of the K2 test; skewness close to 0 as a second criterion)
candidates = results.drop(index="Raw (no transformation)")
best_all = candidates["K2 p-value"].idxmax()
best_simple = candidates.loc[SIMPLE, "K2 p-value"].idxmax()
print(f"Best overall (highest K2 p-value): {best_all}  (p = {candidates.loc[best_all, 'K2 p-value']:.4f})")
print(f"Best among the simple transformations: {best_simple}  (p = {candidates.loc[best_simple, 'K2 p-value']:.4f})")

# Before / after plots
show = ["Raw (no transformation)", "Natural log", "Square root", best_simple]
show = list(dict.fromkeys(show))
fig, axes = plt.subplots(2, len(show), figsize=(4 * len(show), 7))
for j, name in enumerate(show):
    v = transformed[name]
    axes[0, j].hist(v, bins="auto", density=True, color="lightsteelblue", edgecolor="white")
    grid = np.linspace(v.min(), v.max(), 300)
    axes[0, j].plot(grid, stats.norm.pdf(grid, v.mean(), v.std(ddof=1)), color="tab:red", linewidth=2)
    axes[0, j].set_title(f"{name}\nK2 p = {results.loc[name, 'K2 p-value']:.3g}", fontsize=10)
    stats.probplot(v, dist="norm", plot=axes[1, j]); axes[1, j].set_title("Q-Q plot", fontsize=10)
plt.tight_layout(); plt.show()

**i) Which transformation improved normality the most?**
- The **natural logarithm** over-corrects: the strong right skew turns into a left skew (≈ −0.93) and the K² test still rejects normality (p ≈ 0.019; Shapiro–Wilk p ≈ 0.005).
- The **square root** is not enough to remove the skewness (≈ 0.57), although the K² test no longer rejects (p ≈ 0.107).
- The **cube root** is the best of the simple transformations: skewness ≈ 0.06, excess kurtosis ≈ 0.11, K² p ≈ 0.91, Shapiro–Wilk p ≈ 0.54. The log(1 + x) transformation is also good (K² p ≈ 0.67).
- Among all transformations, **Yeo–Johnson** (λ ≈ 0.13, K² p ≈ 0.99) and **Box–Cox** (λ ≈ 0.30, p ≈ 0.95) give the highest p-values. Their λ was estimated from the same data, so their p-values are slightly optimistic; the estimated λ ≈ 0.3 also explains why the cube root (exponent 1/3) works so well and the logarithm (exponent 0) is too strong.

**ii) Is the transformed data now appropriate for parametric tests?** **Yes**, after the cube root (or Box–Cox) transformation the data are compatible with normality: both normality tests do not reject H0, the Q-Q plot is close to the straight line and skewness and kurtosis are close to 0, so t-tests, ANOVA or regression can be applied **on the transformed scale**. Two cautions: (1) the conclusions refer to the transformed variable (a mean of cube-root values is not the mean waiting time, and back-transforming a confidence interval gives an interval for the median-type center, not the mean); (2) failing to reject normality with n = 50 does not prove it, but together with the plots it is sufficient for the usual parametric methods, which are fairly robust at this sample size.

## Summary
| Exercise | Result |
|---|---|
| 1 – heights (n = 50) | approximately normal; negligible skewness (−0.14), only minor tail deviations |
| 2 – reaction times (n = 40) | Shapiro–Wilk p ≈ 0.0018 → reject H0 at 0.05 and 0.01; right-skewed, not normal |
| 3 – exam scores A / B | both tests do not reject normality in either group; Shapiro–Wilk is more powerful and more sensitive to the tails; KS with estimated parameters is very conservative |
| 4 – waiting times (n = 50) | raw: skewness 2.13, kurtosis 6.5, K² p ≈ 8·10⁻⁹ → not normal; cube root (or Box–Cox) makes the data normal enough for parametric tests |